In [22]:
import os
from dotenv import load_dotenv, find_dotenv

print(find_dotenv())          # Should print the path to your .env file
load_dotenv(override=True)    # Load the variables into the environment

print(os.getenv("MLFLOW_TRACKING_URI"))       # Should print the .mlflow URL
print(os.getenv("MLFLOW_TRACKING_USERNAME"))  # Should print your username
print("Token set:", bool(os.getenv("MLFLOW_TRACKING_PASSWORD")))  # Should print True

c:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\.env
https://dagshub.com/BABARALIKHAN098/Mlops_project_wine_quality.mlflow
BABARALIKHAN098
Token set: True


In [ ]:
import os
%pwd

'c:\\Users\\H_KhaN\\OneDrive\\Desktop\\Mlops_project_wine_quality\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\H_KhaN\\OneDrive\\Desktop\\Mlops_project_wine_quality'

In [12]:
from dataclasses import dataclass  # Decorator that auto-generates __init__, __repr__, etc.
from pathlib import Path           # Used for file and folder paths


# frozen=True makes the config immutable, so values can't be changed after creation
@dataclass()
class ModelEvaluationConfig:
    """Holds all settings needed for the model evaluation stage."""

    root_dir: Path            # Root folder where evaluation outputs are stored
    test_data_path: Path      # Path to the test dataset (e.g., test.csv)
    model_path: Path          # Path to the trained model file (e.g., model.joblib)
    all_params: dict          # Model hyperparameters (e.g., from params.yaml)
    metric_file_name: Path    # File where evaluation metrics (e.g., JSON) are saved
    target_column: str        # Name of the target/label column in the dataset
    mlflow_uri: str           # MLflow tracking server URI for logging experiments        # MLflow tracking server URI for logging experiments
    

In [16]:
from src.Mlops_project_quality_predictor.constants import *
from src.Mlops_project_quality_predictor.utils.common import read_yaml,create_directories,load_bin,save_bin,save_json

In [23]:
import os                      # Used to read the MLflow URI from an environment variable
from pathlib import Path       # Used to convert string paths from YAML into Path objects


class ConfigurationManager:
    """
    Central place to load project configuration and build typed config
    objects (e.g., ModelEvaluationConfig) for each pipeline stage.
    """

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,   # Path to config.yaml (paths, directories, artifacts)
        params_filepath=PARAMS_FILE_PATH,   # Path to params.yaml (model/hyperparameters)
        schema_filepath=SCHEMA_FILE_PATH,   # Path to schema.yaml (expected columns & dtypes)
    ):
        # Read each YAML file and store it as an attribute.
        # read_yaml returns a ConfigBox so keys can be accessed
        # with dot notation (e.g., self.config.model_evaluation)
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        # Make sure the top-level artifacts folder exists before any stage runs
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        """Build and return the config object for the model evaluation stage."""

        config = self.config.model_evaluation   # Paths/files for this stage (config.yaml)
        params = self.params.ElasticNet         # Model hyperparameters (params.yaml)
        schema = self.schema.TARGET             # Target column info (schema.yaml)

        # Read the tracking URI from the environment (loaded from .env via load_dotenv).
        # It must be the DagsHub URL ending in .mlflow, never the token.
        mlflow_uri = os.getenv("MLFLOW_TRACKING_URI")
        if not mlflow_uri or not mlflow_uri.startswith("http"):
            raise ValueError(
                "MLFLOW_TRACKING_URI is missing or is not a URL. "
                "Run load_dotenv() before creating the config."
            )

        # Create the evaluation output folder if it doesn't exist yet
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),                    # Folder for evaluation outputs
            test_data_path=Path(config.test_data_path),        # Test dataset location
            model_path=Path(config.model_path),                # Trained model location
            all_params=params,                                 # Hyperparameters to log
            metric_file_name=Path(config.metric_file_name),    # File where metrics are saved
            target_column=schema.name,                         # Name of the label column
            mlflow_uri=mlflow_uri,                             # DagsHub MLflow URL from the environment
        )

        return model_evaluation_config

In [24]:
import os 
import pandas as pd
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib
import json

In [25]:
class ModelEvaluation:
    """Evaluates the trained model on test data and logs results to MLflow."""

    def __init__(self, config: ModelEvaluationConfig):
        # Store the config so every method can access paths and params
        self.config = config

    def eval_metrics(self, actual, pred):
        """Return RMSE, MAE and R2 for the given actual and predicted values."""
        rmse = np.sqrt(mean_squared_error(actual, pred))  # Root mean squared error
        mae = mean_absolute_error(actual, pred)           # Mean absolute error
        r2 = r2_score(actual, pred)                       # Coefficient of determination
        return rmse, mae, r2

    def log_into_mlflow(self):
        """Load the model and test data, compute metrics, save them locally, and log to MLflow."""

        # Load the test dataset and the trained model
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        # Split features and target (columns=[...] is the safe way; axis=-1 is invalid)
        test_x = test_data.drop(columns=[self.config.target_column])
        test_y = test_data[self.config.target_column]

        # Point MLflow to the tracking server (a URL, not a token)
        mlflow.set_tracking_uri(self.config.mlflow_uri)

        # Find out if the tracking store is a local "file" store or a remote server
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        # Start a new MLflow run (the method is start_run, not start.run)
        with mlflow.start_run():
            # Predict on the test features
            predicted_qualities = model.predict(test_x)

            # Calculate evaluation metrics
            rmse, mae, r2 = self.eval_metrics(test_y, predicted_qualities)

            # Save metrics locally as JSON
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            # Log hyperparameters and metrics to MLflow
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            # The model registry does not work with a local file store,
            # so only register the model when using a remote server
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNet")
            else:
                mlflow.sklearn.log_model(model, "model")

In [26]:
try:
    # Load config.yaml, params.yaml and schema.yaml
    config = ConfigurationManager()

    # Build the typed config object for the model evaluation stage
    model_evaluation_config = config.get_model_evaluation_config()

    # Create the evaluator class (ModelEvaluation), NOT the config dataclass
    model_evaluation = ModelEvaluation(config=model_evaluation_config)

    # Run evaluation: compute metrics, save them as JSON, log everything to MLflow
    model_evaluation.log_into_mlflow()

except Exception as e:
    # Print the error for quick debugging, then re-raise it
    # so the pipeline still stops and the full traceback is kept
    print(f"Model evaluation failed: {e}")
    raise

[2026-10-16 22:20:31,147:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\config\config.yaml]
[2026-10-16 22:20:31,152:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\params.yaml]
[2026-10-16 22:20:31,157:INFO:common:YAML file loaded successfully: C:\Users\H_KhaN\OneDrive\Desktop\Mlops_project_wine_quality\schema.yaml]
[2026-10-16 22:20:31,159:INFO:common:created directory at: artifacts]
[2026-10-16 22:20:31,162:INFO:common:created directory at: artifacts/model_evaluation]
[2026-10-16 22:20:40,459:INFO:common:json file saved at: artifacts\model_evaluation\metrics.json]


2026/10/16 22:20:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'ElasticNet'.
2026/10/16 22:21:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNet, version 1
Created version '1' of model 'ElasticNet'.


🏃 View run omniscient-croc-505 at: https://dagshub.com/BABARALIKHAN098/Mlops_project_wine_quality.mlflow/#/experiments/0/runs/ce7f0dd96b194c718da58b103131ab51
🧪 View experiment at: https://dagshub.com/BABARALIKHAN098/Mlops_project_wine_quality.mlflow/#/experiments/0
